# s0: Build City Distance Matrices
Reads `data/cities.csv`, calls the OpenRouteService distance matrix API, and writes:
- `data/adjacencyMatrixDist.csv` — driving distances in **miles**
- `data/adjacencyMatrixTravelTime.csv` — driving durations in **minutes**
- `data/city_distances.csv` — same data in long (pairwise) format

Both matrix files use city names as the row and column index, matching the format expected by `s2_probDefAndRPM.ipynb`.

## Configuration

In [1]:
import importlib.util, pathlib

_spec = importlib.util.spec_from_file_location(
    "api_keys",
    pathlib.Path("../../api_keys.py").resolve()
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
ORS_API_KEY = _mod.ORS_API_KEY

CITIES_CSV   = '../../data/cities.csv'
OUT_DIST_MAT = '../../data/adjacencyMatrixDist.csv'
OUT_TIME_MAT = '../../data/adjacencyMatrixTravelTime.csv'

KM_TO_MILES = 0.621371

## Load cities

In [2]:
import pandas as pd
import numpy as np

cities = pd.read_csv(CITIES_CSV)
display(cities)

,city,lat,long,pop,svi
0,Ada,40.768056,-83.825278,5334,0.390000
1,Alger,40.709722,-83.844167,837,0.594200
2,Bluffton,40.889444,-83.879167,3967,0.450800
3,Cairo,40.830833,-84.084444,517,0.190000
4,Caledonia,40.636389,-82.969444,560,0.327500
5,Carey,40.949444,-83.376111,3565,0.414200
6,Columbus Grove,40.920556,-84.059722,2160,0.350800
7,Continental,41.100278,-84.272500,1102,0.399200
8,Cridersville,40.664722,-84.130833,1791,0.676700
9,Delphos,40.861111,-84.350000,7117,0.562500


## Build ORS coordinate list
ORS expects `[longitude, latitude]` order.

In [3]:
# ORS distance_matrix expects [longitude, latitude]
coordinates = cities[['long', 'lat']].values.tolist()
city_names  = cities['city'].tolist()

print(f"{len(city_names)} cities loaded")
print(list(zip(city_names, coordinates))[:5])

53 cities loaded
[('Ada', [-83.825278, 40.768056]), ('Alger', [-83.844167, 40.709722]), ('Bluffton', [-83.879167, 40.889444]), ('Cairo', [-84.084444, 40.830833]), ('Caledonia', [-82.969444, 40.636389])]


## Call ORS distance matrix API

In [4]:
import openrouteservice as ORS

client = ORS.Client(key=ORS_API_KEY)

n = len(city_names)
indices = list(range(n))

response = client.distance_matrix(
    locations=coordinates,
    profile='driving-car',
    metrics=['distance', 'duration'],
    sources=indices,
    destinations=indices,
    units='km'
)

print("API call successful")

API call successful


## Build adjacency matrices

In [5]:
raw_distances = response['distances']   # km
raw_durations = response['durations']   # seconds

dist_miles   = np.array(raw_distances) * KM_TO_MILES          # km  → miles
time_minutes = np.array(raw_durations) / 60.0                  # sec → minutes

adj_dist = pd.DataFrame(dist_miles.round(2),   index=city_names, columns=city_names)
adj_time = pd.DataFrame(time_minutes.round(0).astype(int), index=city_names, columns=city_names)

adj_dist.index.name = 'city'
adj_time.index.name = 'city'

display("Distance matrix (miles):")
display(adj_dist)
display("Travel time matrix (minutes):")
display(adj_time)

'Distance matrix (miles):'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Marysville,Plain City,Richwood,Milford Center,Findlay,Fostoria,McComb,Arlington,Rawson,Arcadia
city,,,,,,,,,,,,,,,,,,,,,
Ada,0.00,5.29,11.58,19.42,63.57,41.11,22.67,44.50,24.87,34.05,...,55.15,69.09,42.28,57.23,24.46,40.19,24.98,18.03,14.09,32.53
Alger,5.24,0.00,14.58,22.42,68.59,46.14,25.67,47.50,22.89,37.05,...,50.06,63.99,42.38,52.14,29.48,45.22,30.01,23.06,19.11,37.56
Bluffton,11.44,16.47,0.00,16.40,66.72,34.18,11.20,33.17,26.71,31.04,...,66.32,80.26,55.72,68.40,18.15,33.89,18.86,12.16,7.87,26.23
Cairo,19.53,24.56,16.92,0.00,74.81,48.96,6.87,28.25,15.25,16.37,...,66.44,80.37,63.81,68.52,32.94,48.67,33.62,29.28,22.66,41.02
Caledonia,63.65,68.67,66.64,74.48,0.00,40.61,77.73,99.56,86.04,89.11,...,43.83,45.84,24.68,50.79,54.64,55.87,66.90,55.54,64.06,60.38
Carey,40.15,45.18,34.05,48.73,39.88,0.00,44.74,56.31,59.04,63.37,...,60.97,73.21,48.41,67.03,17.40,16.19,29.65,23.46,26.82,17.98
Columbus Grove,22.75,27.78,11.20,6.87,78.04,44.86,0.00,21.41,21.34,22.46,...,72.53,86.46,67.03,74.61,24.37,40.12,23.61,22.85,17.06,32.47
Continental,44.40,49.42,33.17,28.25,99.68,56.58,21.41,0.00,38.31,21.79,...,89.78,103.72,88.68,91.86,37.44,49.39,26.96,44.81,34.21,45.21
Cridersville,24.66,22.64,26.91,15.28,87.03,58.95,21.37,38.35,0.00,27.96,...,62.56,76.50,64.59,64.64,42.93,58.66,43.61,36.97,32.65,51.01


'Travel time matrix (minutes):'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Marysville,Plain City,Richwood,Milford Center,Findlay,Fostoria,McComb,Arlington,Rawson,Arcadia
city,,,,,,,,,,,,,,,,,,,,,
Ada,0,10,19,25,71,46,32,66,40,38,...,69,83,65,71,34,60,43,24,27,47
Alger,10,0,27,33,79,55,40,74,39,46,...,60,74,65,62,42,68,51,32,35,55
Bluffton,18,27,0,20,74,38,21,53,34,34,...,86,100,80,88,21,48,28,20,15,34
Cairo,25,33,21,0,80,54,11,45,31,18,...,88,102,86,90,38,64,47,33,31,51
Caledonia,71,79,73,79,0,47,87,120,98,92,...,57,60,38,63,64,71,79,62,80,73
Carey,47,55,37,54,48,0,57,81,67,67,...,83,86,63,89,23,26,39,29,40,30
Columbus Grove,33,41,21,11,88,58,0,34,40,27,...,97,111,94,99,40,66,37,40,31,53
Continental,66,74,53,45,121,82,34,0,70,36,...,127,141,127,129,62,83,46,72,58,74
Cridersville,39,39,34,31,99,67,40,70,0,42,...,79,93,97,81,51,77,60,50,44,64


## Save outputs

In [6]:
adj_dist.to_csv(OUT_DIST_MAT)
adj_time.to_csv(OUT_TIME_MAT)
print(f"Saved: {OUT_DIST_MAT}")
print(f"Saved: {OUT_TIME_MAT}")

Saved: ../../data/adjacencyMatrixDist.csv
Saved: ../../data/adjacencyMatrixTravelTime.csv
